# Magnitude Validation and Conditional Power

Reproduce the fixed difference/sum intersection-union magnitude candidate: five fresh templates x 2,129 inputs (10,645 total), and eight saved-power cells x 2,000 inputs (16,000 total). Public source generates the native design, preserves the original seeds and fitting rules, calculates model-based and participant-CR2 eight-contrast references, independently checks them, and separately reduces their compact events.

This is a **new public reproduction**, not historical file or binary identity. The candidate is not adopted for human inference. This notebook does not include the separate joint-inference candidate or coverage-method exploration. All scientific execution is disabled by `RUN = False`; `FULL = False` selects one replicate per case for a small entry check, not a new calibration result.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
if sys.version_info[:3] != (3, 10, 9):
    raise RuntimeError('Select the reference Python 3.10.9 kernel before installation.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT),
                'numpy==1.23.5', 'scipy==1.10.0', 'pandas==2.0.3', 'matplotlib==3.7.5'], check=True)
sys.path.insert(0, str(ROOT))
from src.tutoring_statistics import reproduction
reproduction.numerical_runtime()


## Reference Environment

Use Python **3.10.9**, NumPy **1.23.5**, SciPy **1.10.0**, R **4.2.2**, Node **22** (reference 22.14.0), and GNU coreutils `sha256sum` on PATH. A POSIX environment provides advisory locks and atomic hard links. The first code cell installs the Python dependencies; R, Node, coreutils and compatible C/C++/Fortran tools are separate prerequisites.

Use the [reference R installation recipe](README.md#reference-environment) with the exact package versions in [conditional_statistics_environment.json](../../configs/conditional_statistics_environment.json), matching packaged `src/tutoring_statistics/data/numerical_environment.json`. In particular, retain Matrix 1.6-5, lme4 1.1-35.5, lmerTest 3.1-3 and clubSandwich 0.7.0. Select the installed R 4.2.2 executable through `RSCRIPT` and expose a separate matching library through `R_LIBS_USER` when needed. The runners check the reference runtime; a version list does not certify identical numerical binaries or a freshly tested installation.

In [ ]:
import json
import shlex
from src.tutoring_statistics import magnitude_design

RUN = False
FULL = False
PRODUCE_POWER_INPUTS = True  # Set False to reuse existing compatible public power outputs.
POWER_BATCH_SIZE = 100  # Each conditional-statistics invocation selects at most 100 sparse IDs.
WORKERS = 1
RSCRIPT = 'Rscript'
NODE = 'node'
STUDY_ROOT = ROOT / 'results' / 'magnitude_study'
POWER_ROOT = ROOT / 'results' / 'magnitude_power_inputs'
FRESH_OUTPUT = STUDY_ROOT / 'fresh_validation'
SAVED_OUTPUT = STUDY_ROOT / 'saved_power_core'
REDUCTION_LABEL = 'full_request' if FULL else 'first_rep_entry'
FRESH_REPORT = FRESH_OUTPUT / ('aggregate_' + REDUCTION_LABEL + '.json')
SAVED_REPORT = SAVED_OUTPUT / ('aggregate_' + REDUCTION_LABEL + '.json')

if not 1 <= POWER_BATCH_SIZE <= 100:
    raise ValueError('POWER_BATCH_SIZE must be between 1 and 100.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS must be a positive integer.')
fresh_design = magnitude_design.manifest('fresh_validation')
saved_design = magnitude_design.manifest('saved_power_core')
fresh_selector = ['--all'] if FULL else ['--start', '0', '--count', '5']
saved_selector = ['--all'] if FULL else ['--start', '0', '--count', '8']
power_ordinals = saved_design['expectedOrdinals'] if FULL else saved_design['expectedOrdinals'][:8]

def command(args, execute=False):
    argv = [str(arg) for arg in args]
    print(shlex.join(argv))
    if execute:
        subprocess.run(argv, cwd=ROOT, check=True)

for specification in (fresh_design, saved_design):
    print(specification['mode'], 'cases=', len(specification['cases']),
          'B=', specification['B'], 'fixed total=', specification['expectedTotal'])
print('Power inputs selected:', len(power_ordinals), 'first:', power_ordinals[:8], 'last:', power_ordinals[-8:])
print('RUN:', RUN, 'FULL:', FULL, 'PRODUCE_POWER_INPUTS:', PRODUCE_POWER_INPUTS)
print('Exact R package pins:', json.loads((ROOT / 'configs/conditional_statistics_environment.json').read_text())
      ['statistical_reference']['R_packages'])


## Fixed Scientific Settings and Selectors

Initial-need amount slopes use the exact native 27-session joint design: nine whole ten-trial schedules per arm. The Gaussian reference has intercept 60, random-intercept SD 8, residual SD 12, 193 recruits per arm, and independent whole-person eligibility probability .8. Actual retained counts are used; 154.4 is an expectation, not a fixed sample size.

Fresh templates are V1, V2, V3, V4, V6 in that order, with B=2,129 each. Retention/design/response seeds have bases 1102610260/1202610260/1302610260 plus `100000*caseIndex+repID`. V6 keeps caseIndex 6. The saved roster selects original power cells 2,4,6,8,10,12,14,16, B=2,000 each, at sparse ordinal `40*(repID-1)+cellIndex-1`. Original power seed bases 802610200/902610200/1002610200 and stride 10000 are unchanged. `magnitude_design` and `power_reference` supply the complete coefficient settings; the notebook does not duplicate the generator or truth calculation.

For `reproduce_magnitude_study.py`, `--all` means the full selected mode, while `--start/--count` slices positions in its ordered roster. Thus saved `--start 0 --count 8` selects IDs 1,3,5,7,9,11,13,15, not IDs 0 through 7. Its alternative `--ids PATH.json` reads a JSON list, for example `[1,3,5,7,9,11,13,15]`; it does **not** accept a comma string. In contrast, the conditional-statistics input producer below requires comma-separated `--ids`. Do not interchange these selectors.

Run the settings cell again after changing switches. For the complete clean-checkout route set `RUN=True`, `FULL=True`, `PRODUCE_POWER_INPUTS=True` and execute the remaining cells in order. This requests expensive formal reproductions; leaving the switches off only prints their commands. B and the planned denominators never shrink for a small entry selection.

## Fresh Validation: Produce, Fit and Independently Read

The magnitude runner generates each selected original fresh input, commits the request and retention metadata, fits through the unchanged mixed-model kernels, and computes all eight D/S references plus both named magnitude methods. The independent path replays the input law and checks actual RDS/frame quantities, covariance, contrast-specific degrees of freedom, tails and composition. A completed per-input commitment is distinct from the final full-roster reduction.

For each of A12/B12/A13/B13, `D=a-b` and `S=a+b`; `abs(a)<abs(b)` is equivalent to `D*S<0`. The original candidate combines the two fixed directional intersections with Bonferroni2, then applies Holm4 with strict adjusted p<.05. The model-based diagnostic is retained separately from participant CR2. No joint-Gaussian replacement, sign-selected tail or changed guard is used.

In [ ]:
fresh_command = [sys.executable, ROOT / 'scripts/reproduce_magnitude_study.py',
                 '--mode', 'fresh_validation', '--output', FRESH_OUTPUT,
                 '--rscript', RSCRIPT, '--node', NODE, '--workers', WORKERS,
                 '--execute', *fresh_selector]
command(fresh_command, execute=RUN)


## Saved Power: Create the Sparse Core or Reuse Existing Fits

A clean checkout has no saved responses or fits. The cell below first creates exactly the requested original power inputs through `reproduce_conditional_statistics.py`: full mode selects the **16,000 sparse core IDs only**, in batches of at most 100. It does not request `--full` on the 80,000-input parent grid, use a contiguous approximation, or require private cached results. Compatible committed inputs are reused on resume rather than redrawn. Every call uses `--no-aggregate`; the parent power manifest still declares 80,000 planned inputs, but its per-input commitments are sufficient for the selected-core adapter. A whole-80k aggregate is neither required nor claimed.

To reuse existing compatible, authenticated public power outputs (including a complete 80,000-input reproduction), set `PRODUCE_POWER_INPUTS=False` and `POWER_ROOT` to the output root **containing** `power/manifest.json`, `power/events/` and the numerical fits. Keep that tree and its bound sources/artifacts unchanged. This is not an adapter for arbitrary private historical result trees. Missing or changed committed inputs are refused rather than silently fitted by the magnitude analysis.

The existing input producer preserves its original precision/input/fit/readback rules. New input production is separate from the subsequent saved-fit magnitude analysis: the latter adds no independent power sample and does not refit the saved models.

In [ ]:
power_batches = [power_ordinals[start:start + POWER_BATCH_SIZE]
                 for start in range(0, len(power_ordinals), POWER_BATCH_SIZE)]
power_commands = [
    [sys.executable, ROOT / 'scripts/reproduce_conditional_statistics.py', 'run',
     '--execute', '--family', 'power', '--ids', ','.join(map(str, ids)),
     '--batch-size', POWER_BATCH_SIZE, '--no-aggregate', '--output', POWER_ROOT,
     '--rscript', RSCRIPT, '--node', NODE]
    for ids in power_batches
]
print(len(power_commands), 'bounded commands for', len(power_ordinals), 'exact sparse IDs.')
if RUN and PRODUCE_POWER_INPUTS:
    for index, args in enumerate(power_commands, 1):
        print('Power input batch', index, 'of', len(power_commands))
        command(args, execute=True)
else:
    print('Input production disabled. Showing first/last planned commands; all are in power_commands.')
    command(power_commands[0])
    if len(power_commands) > 1:
        command(power_commands[-1])


## Analyze the Saved Fits

The same magnitude runner admits each selected original public power case, response, retained-person mask, fit and commitment. It adds the required eight-L magnitude analysis and independent package readback while preserving the original baseline and CR2 guards. The sparse case/rep identities and the originally generated coefficients determine magnitude truth; historical signed `trueNullIDs` must not replace that calculation. In S6, the B slope crosses zero, so multiplier is not a globally monotone magnitude-effect scale.

In [ ]:
saved_command = [sys.executable, ROOT / 'scripts/reproduce_magnitude_study.py',
                 '--mode', 'saved_power_core', '--output', SAVED_OUTPUT,
                 '--power-directory', POWER_ROOT, '--rscript', RSCRIPT, '--node', NODE,
                 '--workers', WORKERS, '--execute', *saved_selector]
command(saved_command, execute=RUN)


## Separate Producer and Independent Reductions

After all selected members finish, `--aggregate` admits the retained commitments and calls the producer and independently written scalar reducers. It is an aggregate-only invocation: do not combine it with `--all`, `--ids` or `--start/--count`. Full completion requires the exact fixed roster, not a sum of job counts. Fresh and saved modes write five and eight case reports respectively; partial selections retain the original B and explicit missing/technical denominators. The producer uses beta quantiles and the independent reducer uses `betaincinv` for pointwise central 95% Clopper-Pearson bounds.

Each report preserves planned M, technical T, comparison availability Aj, all-four availability A4, full and available-only events, paired method transitions, retained-count distributions and all eight linear coverage/tail diagnostics. Statistical unavailability is not technical uncertainty. Available comparisons may reject even when A4 is false; there is no A4-only discovery gate. R is any rejection, F any true-null rejection, and D any correct alternative discovery. Inapplicable F or D is null, not a measured zero.

In [ ]:
command([sys.executable, ROOT / 'scripts/reproduce_magnitude_study.py',
         '--mode', 'fresh_validation', '--output', FRESH_OUTPUT, '--rscript', RSCRIPT,
         '--node', NODE, '--execute', '--aggregate', FRESH_REPORT], execute=RUN)
command([sys.executable, ROOT / 'scripts/reproduce_magnitude_study.py',
         '--mode', 'saved_power_core', '--output', SAVED_OUTPUT, '--power-directory', POWER_ROOT,
         '--rscript', RSCRIPT, '--node', NODE, '--execute', '--aggregate', SAVED_REPORT], execute=RUN)


## Inspect the Calculated Reports

The cell below displays stored reduction fields only; it does not calculate p-values, coverage intervals or replacement denominators in the notebook. Read `complete`, `completedTotal`, `plannedTotal` and `independentReductionAgreement` together. A filename containing `full_request` is not evidence of completion. The family table includes both named methods and both denominator conventions; the linear table retains all D/S coverage findings, including unfavorable ones. Full JSON also retains contrast-level Aj, tails, distributions and paired transitions.

In [ ]:
import pandas as pd
from IPython.display import display

family_rows, coverage_rows = [], []
for mode, path in (('fresh_validation', FRESH_REPORT), ('saved_power_core', SAVED_REPORT)):
    if not path.is_file():
        print(mode, 'has no report at', path, '; no result is inferred.')
        continue
    summary = json.loads(path.read_text())
    print(mode, {key: summary[key] for key in (
        'completedTotal', 'plannedTotal', 'complete', 'independentReductionAgreement',
        'methodAdoptedForHumanData', 'establishes80PercentMDE', 'establishesHumanSampleSize')})
    for case in summary['results']:
        for method_name, method in case['methods'].items():
            for event_name, family in method['family'].items():
                full = family['all'] if family is not None else {}
                available = family['allAvailable'] if family is not None else {}
                family_rows.append(dict(mode=mode, case=case['caseID'], method=method_name,
                    event=event_name, applicable=family is not None, M=method['M'], T=method['T'],
                    A4=method['A4'], missing=method['missing'], technical_unresolved=method['technicalUnresolved'],
                    count=full.get('count'), full_rate=full.get('fullRate'), full_CI=full.get('fullCI'),
                    completed_rate=full.get('completedRate'), unknown_event_bounds=full.get('unknownEventBounds'),
                    available_count=available.get('count'), available_denominator=available.get('denominator'),
                    available_rate=available.get('rate'), available_CI=available.get('CI')))
            for linear_id, reference in method['linearReferences'].items():
                full, available = reference['coverageAll'], reference['coverageAvailable']
                coverage_rows.append(dict(mode=mode, case=case['caseID'], method=method_name,
                    linear_id=linear_id, M=full['M'], T=full['T'], available=reference['available'],
                    covered=full['count'], full_rate=full['fullRate'], full_CI=full['fullCI'],
                    available_rate=available['fullRate'], available_CI=available['fullCI']))
if family_rows:
    display(pd.DataFrame(family_rows))
if coverage_rows:
    display(pd.DataFrame(coverage_rows))


## Interpretation, Resume and Evidence Limits

The historical completed candidate results had F=0/2,129 for all five fresh templates; V6 also had D=0/2,129, with A4=2,128 rather than 2,129. Saved-power candidate discovery counts, in core-cell order, were 0,0,1,475,0,0,0,315 out of 2,000 each. Peak discovery was 23.75% for S5 multiplier4 and 15.75% for S6 multiplier4. These are historical observations for comparison, not values calculated by opening this notebook, forced success criteria or evidence of human effect sizes.

Retain the adverse S5 multiplier2 A12:S and A13:S linear coverage results: each 1,879/2,000=.9395, with pointwise upper confidence bound .9495479849066164. The older S1 B-difference .9395 is a separate result preserved by [saved_coverage_and_contrasts.ipynb](saved_coverage_and_contrasts.ipynb), not an extra core-power case or a replaced finding. D/S intervals are not intervals for absolute-magnitude differences. Low family rejection does not repair these intervals or establish universal constituent validity.

No tested point established 80% discovery; no 80% MDE, minimum N, or adequacy/inadequacy of 193 human recruits follows. These are finite, assumption-conditional Gaussian results, with simulated whole-person retention and nuisance parameters, not a final human-analysis choice. Saved-fit power is retrospective and is not independent fresh validation of a method chosen after seeing related data. Pointwise Monte Carlo bounds are not simultaneous guarantees.

Compatible resumes keep the same output roots, scientific settings, sources and bound input trees. Completed inputs, fits and magnitude commitments are reused; incomplete or conflicting artifacts are preserved and refused, not silently redrawn or overwritten. Use a distinct reduction filename (change `REDUCTION_LABEL` and rerun settings) when reducing a different completed roster. Existing reports displayed above are not reverified merely by reading JSON: rerun the aggregate command when renewed admission and reduction are required.

The generated manifest, sources, native inputs, per-unit input/result/commit files and both calculated case-report families belong under ignored `results/`. Do not discard their dependencies after extracting a table. Source-calling coverage, selected-entry checks, a newly completed full reproduction and historical scientific acceptance remain distinct; this unexecuted notebook alone establishes none of their numerical outcomes.